# Mapping Surface Runoff Susceptibility in the Nisqually River Watershed, Washington
## Using a Fuzzy Logic Model Under Historical and Future Climate Conditions

**Author:** Nym Griggs  
**Course:** Earth Data Analytics Capstone - University of Colorado Boulder  
**Advisor:** Dr. Lilly Jones  
**Date:** July 2026  

---

## Project Overview

This notebook maps surface runoff susceptibility across the Nisqually River Watershed using a fuzzy logic modeling framework. The analysis integrates topography, soils, land cover, and climate data to produce spatially continuous susceptibility maps under both historical (1976–2005) and projected future (2041–2070) climate conditions.

The resulting susceptibility maps are intended to serve as a watershed-scale screening tool for identifying areas that may be more susceptible to surface runoff and how that susceptibility may shift with climate change. This fuzzy logic model was selected as it is less resource and data intensive than many physically based hydrologic models, supporting targeted resource allocation and prioritization of high-risk areas. It is not intended to replace detailed hydrologic modeling, but rather offer a complementary tool to provide an accessible first-pass watershed-scale assessment when time, data, or computational resources are limited.

**Research Questions**

1. Where is surface runoff susceptibility highest in the Nisqually River Watershed under historical conditions?
2. Do projected mid-century increases in wet-season precipitation and storm intensity redistribute that risk, or simply intensify it where it already exists?

---
## Methods

Surface runoff susceptibility was modeled using a fuzzy logic framework that integrates topographic, impervious surface, hydrologic soil group, and climate variables to estimate the relative likelihood of surface runoff generation and concentration across the Nisqually River Watershed. The topographic inputs such as slope were derived from a Digital Elevation Model (DEM) and flow accumulation was calculated using the `pysheds` [Python library](https://github.com/pysheds/pysheds). 

From the MACA climate dataset, mean wet-season (December–February) precipitation and Rx1day (maximum 1-day precipitation) were calculated. Rx1day was included as a proxy for storm intensity. Historical (1976–2005) and mid-century future (2041–2070) climate scenarios were analyzed using the Community Climate System Model Version 4 (CCSM4) under the Representative Concentration Pathway (RCP) 4.5 emissions scenario. CCSM4 was selected because its projections are representative of the average response across climate models for the region. The mid-century future period was chosen to balance capturing projected climate change impacts while remaining relevant for near- to medium-term planning and decision-making. Although this analysis uses CCSM4, the workflow can be readily adapted to other climate models available through the MACA dataset with minor modifications.

Gamma values ranging from 0.5 to 0.9 were evaluated during model development. The gamma parameter controls the balance between the fuzzy algebraic product (more conservative) and fuzzy algebraic sum (more optimistic). A gamma value of 0.85 was selected because it produced results was selected because it provided a balanced representation of runoff susceptibility while avoiding overly restrictive or overly permissive predictions.

Each model input was transformed into a standardized fuzzy membership value ranging from 0 to 1, where higher values indicate a greater contribution to runoff susceptibility. Runoff generation and runoff concentration variables were modeled separately to better represent hydrological conditions, before being combined to produce the final surface runoff susceptibility map. The fuzzified layers were then combined using a fuzzy gamma overlay (γ = 0.85). The final workflow produced maps of historical runoff susceptibility, projected future runoff susceptibility, and projected changes in runoff susceptibility.

---

## Table of Contents

1. **Getting Started (Setup)**
   - Required Datasets
   - Import Libraries
   - Configure Project Directories

2. **Data Acquisition & Preparation**
   - Study Area
   - Digital Elevation Model (SRTM)
   - Land Cover (NLCD Impervious Surface)
   - Climate Data (MACA)
   - Hydrologic Soil Groups (gSSURGO)

3. **Data Processing**
   - Topography Analysis - Flow Accumulation & Slope
   - Climate Data Processing
   - Data Harmonization

4. **Fuzzy Logic Modeling**
   - Membership Functions
   - Fuzzification
   - Fuzzy Overlay
   - Runoff Susceptibility Model

5. **Results & Maps**
   - Historical Susceptibility
   - Future Susceptibility
   - Change Analysis
   - Interactive Visualization

## 1. Getting Started (Setup)

### 1.1 Required Datasets

Before running the notebook, download the following datasets and place them in the appropriate project folders. The notebook will automatically create the required directory structure, but downloaded datasets must be placed in their corresponding folders on your local machine.

The SRTM Digital Elevation Model (DEM) and MACA climate datasets are accessed and downloaded programmatically within the notebook and do not require manual download.

**Note:** File paths in the notebook may need to be updated to match your local directory structure. Although this workflow is configured for the Nisqually River Watershed (HUC8: 17110015), it can be adapted to other HUC8 watersheds with minor modifications.

##### 1. Watershed Boundary (USGS WBD)
- Source: https://www.usgs.gov/national-hydrography/watershed-boundary-dataset  
- Map Downloader Link: https://apps.nationalmap.gov/downloader/#/
- Download the HUC8 watershed boundary that includes the Nisqually River watershed (HUC8: 17110015)  
- Place files in: watershed-boundary-dataset/


##### 2. NLCD Impervious Surface
- Source: https://www.mrlc.gov/viewer/
- Product: NLCD Impervious Surface (30 m) for Study Area
- Place files in: nlcd-impervious/

##### 3. Soils (NRCS gSSURGO)
- Source: https://www.nrcs.usda.gov/resources/data-and-reports/gridded-soil-survey-geographic-gssurgo-database
- Download gSSURGO for Washington State (geodatabase)
- Place files in: soils/

### 1.2 Import Libraries

In [1]:
# Work with files, folders, and file paths
import os
import pathlib
import zipfile
from pathlib import Path
from glob import glob

# Work with tabular data
import pandas as pd

# Perform numerical and scientific computations
import numpy as np

# Work with vector geospatial data
import geopandas as gpd

# Work with raster and multidimensional geospatial data
import rioxarray as rxr
import xarray as xr
import xrspatial

# Merge raster arrays
from rioxarray.merge import merge_arrays

# Control raster resampling during reprojection
from rasterio.enums import Resampling

# Calculate flow direction and flow accumulation from elevation data
from pysheds.grid import Grid

# Create static plots
import matplotlib.pyplot as plt

# Configure colormaps and color normalization
from matplotlib import colors
import matplotlib.cm as cm

# Create interactive plots from tabular and raster data
import hvplot.pandas
import hvplot.xarray

# Build interactive visualizations
import holoviews as hv

# Create interactive geographic visualizations
import geoviews as gv

# Coordinate reference systems
import cartopy.crs as ccrs

# Access NASA Earthdata products
import earthaccess

# Create fuzzy membership functions and perform fuzzy operations
import skfuzzy as fuzz

C:\Users\nymve\miniconda3\envs\surface-runoff-susceptibility-model\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.3 Configure Project Directories

In [ ]:
# Create base project directory
project_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

# Define all directories in one dictionary
dirs = {
    "project": project_dir,
    "wbd": os.path.join(project_dir, "watershed-boundary-dataset"),
    "topography": os.path.join(project_dir, "topography"),
    "soils": os.path.join(project_dir, "soils"),
    "climate": os.path.join(project_dir, "climate"),
    "impervious": os.path.join(project_dir, "nlcd-impervious"),
    "inputs": os.path.join(project_dir, "final_inputs_5070"), 
    "fuzzy": os.path.join(project_dir, "fuzzy_layers"),      
    "final": os.path.join(project_dir, "final_outputs")
}

# Create all directories
for path in dirs.values():
    os.makedirs(path, exist_ok=True)

# Assign individual dir variables
wbd_dir = dirs["wbd"]
topography_dir = dirs["topography"]
soils_dir = dirs["soils"]
climate_dir = dirs["climate"]
impervious_dir = dirs["impervious"]
fuzzy_dir = dirs["fuzzy"]          
final_dir = dirs["final"]  

# Check one of them
print(wbd_dir)
print(os.path.exists(wbd_dir))

## 2. Data Acquisition & Preparation

### 2.1 Study Area

In [ ]:
# Path to the downloaded zip file
zip_path = os.path.join(wbd_dir, "WBD_17_HU2_GPKG.zip")

# Extracted directory path
extract_dir = os.path.join(wbd_dir, "extracted")
os.makedirs(extract_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Path to GeoPackage (after unzip)
wbd_gpkg_path = os.path.join(extract_dir, "WBD_17_HU2_GPKG.gpkg")

# Read HUC8 layer
wbd_gdf = gpd.read_file(wbd_gpkg_path, layer="WBDHU8")

# Check the first few rows
print(wbd_gdf.head())

In [ ]:
# Check the columns
print(wbd_gdf.columns)

In [ ]:
# Check the geometry type
wbd_gdf.head()

In [ ]:
# Check for the Nisqually watershed
wbd_gdf[wbd_gdf["name"].str.contains("Nisqually", case=False)]

In [ ]:
# Subset to just the Nisqually watershed
nisqually_gdf = wbd_gdf[wbd_gdf["huc8"] == "17110015"]

In [ ]:
# Check the name of the watershed
print(nisqually_gdf["name"])

In [ ]:
# Plot to see that it worked
nisqually_gdf.plot()

In [ ]:
# Plot the results with web tile images
nisqually_gdf_plot = nisqually_gdf.hvplot(
    geo=True, tiles='EsriImagery',
    fill_color=None, line_color='white',
    title='Nisqually Watershed (HUC8)',
    frame_width=500,
    xlabel="Longitude",
    ylabel="Latitude")

# Display the plot with esri imagery
nisqually_gdf_plot

### 2.2 Digital Elevation Model (SRTM - 30 m resolution)

In [ ]:
# Set up Earth Access
earthaccess.login()

In [ ]:
# Search for SRTM data
datasets = earthaccess.search_datasets(keyword = "SRTM DEM")
for dataset in datasets:
    print(dataset['umm']['ShortName'], dataset['umm']['EntryTitle'])

In [ ]:
# File pattern for data
nisqually_gdf_srtm_pattern = os.path.join(topography_dir, '*.hgt.zip')

# Study area for topo data
nisqually_gdf_elev_bounds = tuple(nisqually_gdf.total_bounds)

# Add buffer
buffer = 0.025
nisqually_xmin, nisqually_ymin, nisqually_xmax, nisqually_ymax = nisqually_gdf_elev_bounds
nisqually_elev_bounds_buffer = (nisqually_xmin - buffer,
                                 nisqually_ymin - buffer,
                                 nisqually_xmax + buffer,
                                 nisqually_ymax + buffer)

# Look at the results
srtm_files = glob(nisqually_gdf_srtm_pattern)

if not srtm_files:

    # Search for data
    nisqually_gdf_srtm_search = earthaccess.search_data(
        short_name = 'SRTMGL3',
        bounding_box = nisqually_elev_bounds_buffer
    )

    # Download data
    nisqually_gdf_srtm_results = earthaccess.download(
        nisqually_gdf_srtm_search,
        topography_dir
    )

# Add text if files already downloaded
else:
    print("SRTM files already downloaded")
    nisqually_gdf_srtm_results = srtm_files

In [ ]:
# Check it out
nisqually_gdf_srtm_results

In [ ]:
# Create list to plot DEM
nisqually_gdf_srtm_da_list = []
for srtm_path in glob(nisqually_gdf_srtm_pattern):
    tile_da = rxr.open_rasterio(srtm_path, mask_and_scale = True).squeeze()
    srtm_cropped_da = tile_da.rio.clip_box(*nisqually_elev_bounds_buffer)
    nisqually_gdf_srtm_da_list.append(srtm_cropped_da)

# Merge 
nisqually_srtm_da = merge_arrays(nisqually_gdf_srtm_da_list)

nisqually_srtm_da.plot(cmap='terrain', cbar_kwargs={'label': 'Elevation (meters)'})

nisqually_gdf.boundary.plot(ax = plt.gca(), color='black')

# Add title
plt.title("topo_elevation_nisqually")

# Check it out
plt.show()

In [ ]:
# Save the merged raster to a new file
# Create a copy of the data array to modify for saving
nisqually_save_da = nisqually_srtm_da.copy()

# Remove any existing _FillValue attribute and encoding to avoid issues when writing the raster
nisqually_save_da.attrs.pop("_FillValue", None)

# Remove any existing _FillValue from encoding and set the nodata value to -9999
nisqually_save_da.encoding.pop("_FillValue", None)
nisqually_save_da = nisqually_save_da.rio.write_nodata(-9999)

# Create the path for the new raster file
dem_path = os.path.join(topography_dir, "nisqually_dem.tif")

# Save the raster to a new file
nisqually_save_da.rio.to_raster(dem_path)

# Check that the file was saved correctly
print(dem_path)
print(os.path.exists(dem_path))


In [ ]:
# Check the min and max values
print(nisqually_srtm_da.min().values)
print(nisqually_srtm_da.max().values)

In [ ]:
# Plot with limits for better visualization
nisqually_srtm_da.plot(
    cmap="terrain",
    vmin=0,
    vmax=float(nisqually_srtm_da.max()),
    cbar_kwargs={"label": "Elevation (meters)"}
)

# Add watershed boundary
nisqually_gdf.boundary.plot(ax=plt.gca(), color="black")
plt.title("Nisqually Watershed Elevation (SRTM DEM)")
plt.show()

In [ ]:
# Reproject the SRTM DEM to 5070 to calculate slope later
nisqually_rpj = nisqually_srtm_da.rio.reproject("EPSG:5070")

# Create a copy of the data array to modify for saving
save_da = nisqually_rpj.copy()

# Remove any existing _FillValue attribute and encoding to avoid issues when writing the raster
save_da.attrs.pop("_FillValue", None)

# Remove any existing _FillValue from encoding and set the nodata value to -9999
save_da.encoding.pop("_FillValue", None)
save_da = save_da.astype("float32").fillna(-9999)

# Write the nodata value to the raster metadata
save_da = save_da.rio.write_nodata(-9999)

# Create the path for the new raster file
dem_5070_path = os.path.join(topography_dir, "nisqually_dem_5070.tif")

# Save the raster as float32 with nodata value of -9999
save_da.rio.to_raster(dem_5070_path, dtype="float32")

### 2.3 Land Cover (NLCD Impervious Surface)

In [ ]:
# Path to zip file
zip_path = os.path.join(impervious_dir, "NLCD_84d4ec6e-782d-4ca1-985c-369c3d869faf.zip")

# Unzip to the same directory
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(impervious_dir)

# Check the extracted files
print("Unzipped successfully")

In [ ]:
# List the files in the impervious directory to see the extracted contents
for file in os.listdir(impervious_dir):
    print(file)

In [ ]:
# Find the tif
tif_files = [
    f for f in os.listdir(impervious_dir)
    if f.lower().endswith((".tif", ".tiff"))
]

# Check the tif files found
impervious_path = os.path.join(impervious_dir, tif_files[0])

# Open it
impervious_da = rxr.open_rasterio(impervious_path, masked=True).squeeze()

# Check the dataarray
print(impervious_da)

In [ ]:
# Check CRS, bounds, and resolution
print(impervious_da.rio.crs)
print(impervious_da.rio.bounds())
print(impervious_da.rio.resolution())

In [ ]:
# Reproject impervious to crs 5070
impervious_5070 = impervious_da.rio.reproject("EPSG:5070")

In [ ]:
nisqually_gdf_5070 = nisqually_gdf.to_crs("EPSG:5070")

impervious_clip = impervious_5070.rio.clip(
    nisqually_gdf_5070.geometry,
    nisqually_gdf_5070.crs,
    drop=True
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

impervious_clip.plot(
    ax=ax,
    cmap="viridis",
    cbar_kwargs={"label": "Impervious (%)"}
)

nisqually_gdf_5070.boundary.plot(
    ax=ax,
    edgecolor="white",
    linewidth=1.5
)

ax.set_title("Nisqually Watershed - Impervious Surface")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

In [ ]:
# Create function to clean and save, fill no data
def clean_and_save(da, path):
    """Function to clean the data array and save it to a raster file.
    
    Args:
    da (xarray.DataArray): The data array to clean and save
    path (str): The file path to save the raster to
    
    Returns:
    None
    """
    save_da = da.copy()
    save_da.attrs.pop("_FillValue", None)
    save_da.encoding.pop("_FillValue", None)
    save_da = save_da.astype("float32")
    save_da = save_da.fillna(-9999)
    save_da = save_da.rio.write_nodata(-9999)
    save_da.rio.to_raster(path, dtype="float32")
    print(f"Saved: {path}")

In [ ]:
# Create path to nisqually impervious
impervious_5070_path = os.path.join(
    impervious_dir,
    "nisqually_impervious_5070.tif"
)

# Apply the function
clean_and_save(impervious_clip, impervious_5070_path)

### 2.4 Climate Data (MACA)

In [ ]:
# Create function to convert longitude values
def convert_longitude(longitude):

    """
    Function to convert longitude

    Args:
    longitude:

    Returns:Function to convert longitude
    """
    
    return (longitude - 360) if longitude > 180 else longitude 

In [ ]:
# Create directory for file
climate_pattern = os.path.join(climate_dir, '*.nc')
climate_pattern

In [ ]:
# Define some parameters
site_name = "Nisqually"
site_gdf = nisqually_gdf
date_range = "2041_2045"
model = "CCSM4"
rcp_value = "rcp45"
climate_var = "pr"

In [ ]:
# Create a maca path
climate_path = os.path.join(
    climate_dir,
    f"maca_{model}_{site_name}_{date_range}_CONUS_daily.nc"
)

In [ ]:
# Construct the URL where the climate data lives
climate_url = (
    "http://thredds.northwestknowledge.net:8080/thredds/dodsC"
    "/MACAV2"
    f"/{model}"
    "/macav2metdata"
    f"_{climate_var}"
    f"_{model}_r6i1p1"
    f"_{rcp_value}"
    f"_{date_range}_CONUS"
    "_daily.nc"
)

In [ ]:
# Check out the url
climate_url

In [ ]:
# Open the dataset using xarray
ds = xr.open_dataset(climate_url)
print(ds.data_vars)

In [ ]:
# Check if the variable is "pr" or "precipitation" and assign to climate_da
climate_da = ds["pr"] if "pr" in ds.data_vars else ds["precipitation"]
print(climate_da)

In [ ]:
# Check remote point values to make sure it looks right
pt_raw = climate_da.sel(lat=46.85, lon=237.8, method="nearest")
print("Remote raw point min/max:", float(pt_raw.min()), float(pt_raw.max()))

In [ ]:
# Convert longitude values to the range [-180, 180]
climate_da = climate_da.assign_coords(
    lon=(((climate_da.lon + 180) % 360) - 180).data
).sortby("lon")

In [ ]:
# Check the converted point values to make sure it looks right
pt_conv = climate_da.sel(lat=46.85, lon=-122.2, method="nearest")
print("Remote converted point min/max:", float(pt_conv.min()), float(pt_conv.max()))

In [ ]:
# Match the CRS of the climate data to the watershed boundary
nisqually_4326 = nisqually_gdf.to_crs("EPSG:4326")

# Create a bounding box for the watershed
minx, miny, maxx, maxy = nisqually_4326.total_bounds
print(minx, miny, maxx, maxy)

In [ ]:
# Crop the climate data to the bounding box of the watershed
climate_da_cropped = climate_da.sel(
    lon=slice(minx, maxx),
    lat=slice(miny, maxy)
)

In [ ]:
# Check the cropped point values to make sure it looks right
pt_crop = climate_da_cropped.sel(lat=46.85, lon=-122.2, method="nearest")
print("Remote cropped point min/max:", float(pt_crop.min()), float(pt_crop.max()))

# Check the cropped slice values to make sure it looks right
test_crop = climate_da_cropped.isel(time=0)
print("Remote cropped slice min/max:", float(test_crop.min()), float(test_crop.max()))
print(climate_da_cropped.shape)

In [ ]:
# Load the cropped subset into memory
climate_da_cropped = climate_da_cropped.load()
print("Cropped subset loaded into memory")

In [ ]:
# Create path for cropped file
cropped_path = os.path.join(
    climate_dir,
    "maca_CCSM4_Nisqually_2041_2045_cropped.nc"
)

if not os.path.exists(cropped_path):
    climate_da_cropped.to_netcdf(cropped_path)
    print("Saved cropped file:", cropped_path)
else:
    print("File already exists, not overwriting:", cropped_path)

In [ ]:
# Check the cropped file by opening it again
ds_check = xr.open_dataset(cropped_path)
print(ds_check.data_vars)

# Check if the variable is "pr" or "precipitation" and assign to check_da
check_da = ds_check["precipitation"] if "precipitation" in ds_check.data_vars else ds_check["pr"]

# Check the cropped point values to make sure it looks right
pt_check = check_da.sel(lat=46.85, lon=-122.2, method="nearest")
print("Local cropped point min/max:", float(pt_check.min()), float(pt_check.max()))

# Check the cropped slice values to make sure it looks right
test_check = check_da.isel(time=0)
print("Local cropped slice min/max:", float(test_check.min()), float(test_check.max()))

In [ ]:
# Plot the mean precipitation to check it out
climate_mean = check_da.mean(dim="time")
climate_mean.plot()

# Add title
plt.title("Mean Precipitation")
plt.show()

In [ ]:
# Check the min and max values of the mean precipitation
print(float(climate_mean.min()), float(climate_mean.max()))

In [ ]:
# Check the shape of the cropped data array
climate_da_cropped.shape

#### Function for downloading all maca data cropped precip

In [ ]:
# Create function to get all climate data
# *Created with help of ChatGPT, refined by author*
def get_maca_cropped_precip(
    model,
    scenario,
    date_range,
    watershed_gdf,
    climate_dir,
    site_name="Nisqually",
):
    """
    Function to open a remote MACA daily precipitation dataset for one time chunk,
    convert longitude from 0-360 to -180-180, crop to the watershed
    bounding box, load the cropped daily subset into memory, and save
    it locally as a NetCDF file.

    Args:
    model (str): MACA climate model name, e.g. "CCSM4".
    scenario (str): Scenario name, e.g. "historical", "rcp45", or "rcp85".
    date_range (str): Time chunk string, e.g. "1975_1979" or "2041_2045".
    watershed_gdf (geopandas.GeoDataFrame): Watershed boundary used for cropping.
    climate_dir (str): Directory where the cropped NetCDF file will be saved.
    site_name (str): Name of the study site for use in the output filename. Defaults to "Nisqually".

    Returns
    str: Path to the saved cropped NetCDF file.
    """

    # MACA ensemble convention
    ensemble = "r6i1p1" if model == "CCSM4" else "r1i1p1"

    # Build filename differently for historical vs future scenarios
    if scenario == "historical":
        remote_filename = (
            f"macav2metdata_pr_{model}_{ensemble}_historical_{date_range}_CONUS_daily.nc"
        )
    else:
        remote_filename = (
            f"macav2metdata_pr_{model}_{ensemble}_{scenario}_{date_range}_CONUS_daily.nc"
        )

    # Build remote URL
    climate_url = (
        "http://thredds.northwestknowledge.net:8080/thredds/dodsC"
        f"/MACAV2/{model}/{remote_filename}"
    )

    # Build local output path
    out_path = os.path.join(
        climate_dir,
        f"maca_{model}_{scenario}_{site_name}_{date_range}_cropped.nc"
    )

    # Skip if already exists
    if os.path.exists(out_path):
        print(f"Already exists: {out_path}")
        return out_path

    # Print message about opening remote dataset
    print(f"Opening remote dataset for {scenario} {date_range}...")

    # Open remote dataset
    ds = xr.open_dataset(climate_url)

    # Grab precipitation variable
    climate_da = ds["pr"] if "pr" in ds.data_vars else ds["precipitation"]

    # Convert longitude from 0-360 to -180-180 and sort
    climate_da = climate_da.assign_coords(
        lon=(((climate_da.lon + 180) % 360) - 180).data
    ).sortby("lon")

    # Watershed bounds in EPSG:4326 to match MACA coordinates
    watershed_4326 = watershed_gdf.to_crs("EPSG:4326")
    bounds = watershed_4326.total_bounds

    # Add buffer
    buffer = 0.05
    xmin, ymin, xmax, ymax = bounds
    bounds_buffer = (xmin - buffer, ymin - buffer, xmax + buffer, ymax + buffer)

    # Crop to buffered bounding box
    climate_da_cropped = climate_da.sel(
        lon=slice(bounds_buffer[0], bounds_buffer[2]),
        lat=slice(bounds_buffer[1], bounds_buffer[3])
    )

    # Load cropped subset into memory before saving
    climate_da_cropped = climate_da_cropped.load()

    # Quick sanity check on first day
    test = climate_da_cropped.isel(time=0)
    print("First-day cropped min/max:", float(test.min()), float(test.max()))

    # Save cropped daily data
    climate_da_cropped.to_netcdf(out_path)

    # Close remote dataset
    ds.close()

    print(f"Saved: {out_path}")
    return out_path

In [ ]:
# Make sure function works by running it for one time chunk
test_path = get_maca_cropped_precip(
    model="CCSM4",
    scenario="rcp45",
    date_range="2046_2050",
    watershed_gdf=nisqually_gdf,
    climate_dir=climate_dir,
    site_name="Nisqually"
)

# Check the output path
print(test_path)

In [ ]:
# Define historical date range for looping
historical_ranges = [
    "1975_1979",
    "1980_1984",
    "1985_1989",
    "1990_1994",
    "1995_1999",
    "2000_2004",
    "2005_2005",
]

# Define future date range for looping
future_ranges = [
    "2041_2045",
    "2046_2050",
    "2051_2055",
    "2056_2060",
    "2061_2065",
    "2066_2070",
]

In [ ]:
# Create loop for all historical and future time chunks
saved_files = []

# Loop through historical ranges
for tr in historical_ranges:
    saved_files.append(
        get_maca_cropped_precip(
            model="CCSM4",
            scenario="historical",
            date_range=tr,
            watershed_gdf=nisqually_gdf,
            climate_dir=climate_dir,
            site_name="Nisqually",
        )
    )

# Loop through future ranges
for tr in future_ranges:
    saved_files.append(
        get_maca_cropped_precip(
            model="CCSM4",
            scenario="rcp45",
            date_range=tr,
            watershed_gdf=nisqually_gdf,
            climate_dir=climate_dir,
            site_name="Nisqually",
        )
    )

# Check the list of saved files
saved_files

### 2.5 Hydrologic Soil Groups (gSSURGO/NRCS)

In [ ]:
# Path to the downloaded zip file
soils_zip_path = os.path.join(soils_dir, "gSSURGO_WA.zip")

# Extracted directory path
soils_extract_dir = os.path.join(soils_dir, "extracted")
os.makedirs(soils_extract_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(soils_zip_path, 'r') as zip_ref:
    zip_ref.extractall(soils_extract_dir)

# Path to GeoPackage (after unzip)
soils_gpkg_path = os.path.join(soils_extract_dir, "gSSURGO_WA.gdb")

In [ ]:
# Make sure it exists
soils_gdf = gpd.read_file(soils_gpkg_path)

In [ ]:
# Check the map unit attributes layer
map_unit_attributes = gpd.read_file(soils_gpkg_path, layer="muaggatt")

# Check the first few rows of the map unit attributes
map_unit_attributes.head()

# Check the columns in the map unit attributes
map_unit_attributes.columns

In [ ]:
# Read the map unit aggregate attributes table
map_unit_attributes = gpd.read_file(soils_gpkg_path, layer="muaggatt")
map_unit_attributes.columns = map_unit_attributes.columns.str.lower()

# Read the soils polygon layer
soils = gpd.read_file(soils_gpkg_path, layer="MUPOLYGON")
soils.columns = soils.columns.str.lower()

# Keep only the columns needed for this step
map_unit_attributes_subset = map_unit_attributes[["mukey", "hydgrpdcd"]].dropna()

# Join hydrologic group code to soil polygons
soils_gdf = soils.merge(map_unit_attributes_subset, on="mukey", how="left")

# Simplify hydrologic group values like "A/D" to a single group
def simplify_hydgrp(val):
    if pd.isna(val):
        return None
    return val.split("/")[-1]

# Create a cleaned hydrologic group field
soils_gdf["hydgrp"] = soils_gdf["hydgrpdcd"].apply(simplify_hydgrp)

# Quick checks
print(soils_gdf[["mukey", "hydgrpdcd", "hydgrp"]].head())
print(soils_gdf["hydgrp"].value_counts(dropna=False))

In [ ]:
# Check the CRS of the soils geodataframe
# Keep 5070 for analysis, but reproject to 4326 for plotting
soils_gdf.crs

In [ ]:
# Check out whole map of hydrologic soil groups
soils_gdf.plot(column="hydgrp", legend=True)

In [ ]:
# Check out the columns in the soils geodataframe
print(soils_gdf.columns)

In [ ]:
# Make sure in same CRS for clipping and plotting
soil_gdf_5070 = soils_gdf.to_crs("EPSG:5070")
nisqually_gdf_5070 = nisqually_gdf.to_crs("EPSG:5070")

In [ ]:
# Clip the soil data to the watershed boundary
nisqually_soil_clip = gpd.clip(soil_gdf_5070, nisqually_gdf_5070)

In [ ]:
# Plot the clipped soil data
fig, ax = plt.subplots(figsize=(8,6))

# Plot the soil data with the hydrologic group column and a legend
nisqually_soil_clip.plot(
    column="hydgrp",
    ax=ax,
    legend=True
)

# Add watershed boundary on top
nisqually_gdf_5070.boundary.plot(ax=ax, edgecolor="black")

# Add title
plt.title("Nisqually Watershed - Hydrologic Soil Groups - EPSG:5070")
plt.show()

In [ ]:
# Check the hydrologic group values in the clipped soil data
nisqually_soil_clip["hydgrp"].value_counts(dropna=False)

In [ ]:
# create a clean modeling version
nisqually_soil_clean = nisqually_soil_clip.dropna(subset=["hydgrp"]).copy()

In [ ]:
# Create soil score mapping based on hydrologic group, where A=0.1, B=0.4, C=0.7, D=1.0
soil_score_map = {
    "A": 0.0,
    "B": 0.37,
    "C": 0.78,
    "D": 1.0
}

# Add soil score column to the cleaned soil geodataframe
nisqually_soil_clean["soil_score"] = nisqually_soil_clean["hydgrp"].map(soil_score_map)

# Make sure there are no missing values in the soil score column
nisqually_soil_clean["soil_score"].isna().sum()  

In [ ]:
# Build grid template from the reprojected DEM, for rasterizing the soil polygons below
grid = Grid.from_raster(dem_5070_path)

In [ ]:
# Check the first few rows of the cleaned soil geodataframe
nisqually_soil_polygons = zip(
    nisqually_soil_clean.geometry.values,
    nisqually_soil_clean["soil_score"].values
)

# Rasterize the soil polygons to create a soil score raster
nisqually_soil_raster = grid.rasterize(nisqually_soil_polygons, fill=np.nan)

In [ ]:
# Create path for soil raster file
soil_raster_path = os.path.join(soils_dir, "nisqually_soil_score_5070.tif")

# Save the soil score raster to a GeoTIFF file
grid.to_raster(nisqually_soil_raster, soil_raster_path)

# Print the path to the saved soil raster
print("Saved soil raster to:", soil_raster_path)

In [ ]:
# Open the saved soil raster to check it out
nisqually_soil_da = rxr.open_rasterio(soil_raster_path, masked=True).squeeze()

In [ ]:
# Plot the soil score raster with the watershed boundary on top
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the soil score raster with a colormap and colorbar
nisqually_soil_da.plot(
    ax=ax,
    cmap="viridis",
    cbar_kwargs={"label": "Soil runoff score"} # Label for the colorbar
)

# Add watershed boundary on top with white edges
nisqually_gdf_5070.boundary.plot(
    ax=ax,
    edgecolor="black",
    linewidth=2
)

# Add title
ax.set_title("Nisqually Soil Runoff Score (Rasterized) - EPSG:5070")
plt.tight_layout()
plt.show()

## 3. Data Processing

### 3.1 Topography Analysis - Slope and Flow Accumulation

In [ ]:
# Open the reprojected DEM to check it out
dem_5070 = rxr.open_rasterio(dem_5070_path, masked=True).squeeze()

#### 3.1.1 Slope

In [ ]:
# Calculate slope
nisqually_slope = xrspatial.slope(dem_5070)

In [ ]:
slope_5070_path = os.path.join(topography_dir, "nisqually_slope_5070.tif")

save_da = nisqually_slope.copy()
save_da.attrs.pop("_FillValue", None)
save_da.encoding.pop("_FillValue", None)
save_da = save_da.astype("float32").fillna(-9999)
save_da = save_da.rio.write_nodata(-9999)

save_da.rio.to_raster(slope_5070_path, dtype="float32")

In [ ]:
def reproject_4326(input_path, output_path=None, dst_crs="EPSG:4326"):
    """
    Reproject a raster to a new CRS and save it to a new file for plotting.

    Args:
        input_path (str): Path to the input raster file.
        output_path (str): Path to the output raster file. If None, saves with "_4326" suffix.
        dst_crs (str): Target CRS. Default is "EPSG:4326".

    Returns:
        str: Path to the saved reprojected raster file.
    """

    # Open raster
    da = rxr.open_rasterio(input_path, masked=True).squeeze()

    # Reproject
    da_reprojected = da.rio.reproject(dst_crs)

    # Create output path if not provided
    if output_path is None:
        base, ext = os.path.splitext(input_path)
        base = base.replace("_5070", "")
        output_path = f"{base}_4326{ext}"

    # Clean + standardize before saving
    save_da = da_reprojected.copy()
    save_da.attrs.pop("_FillValue", None)
    save_da.encoding.pop("_FillValue", None)
    save_da = save_da.astype("float32")
    save_da = save_da.fillna(-9999)
    save_da = save_da.rio.write_nodata(-9999)

    # Save
    save_da.rio.to_raster(output_path, dtype="float32")

    print(f"Reprojected raster saved to: {output_path}")
    return output_path

In [ ]:
# Reproject slope to 4326 for plotting
nisqually_slope_4326_path = reproject_4326(slope_5070_path)

# Open the reprojected slope raster to check it out
nisqually_slope_4326 = rxr.open_rasterio(nisqually_slope_4326_path, masked=True).squeeze()

In [ ]:
# Plot slope with watershed boundary
fig, ax = plt.subplots()

# Plot slope with a terrain colormap and colorbar with label
nisqually_slope_4326.plot(
    ax=ax,
    cmap='terrain',
    cbar_kwargs={'label': 'Slope (degrees)'}
)

# Add title
ax.set_title("Nisqually Watershed - Slope (Degrees) - EPSG:4326")

# Plot watershed boundary on top
nisqually_gdf.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1.5)

# Show the plot
plt.show()

#### 3.1.2 Flow Accumulation

In [ ]:
# Read the DEM raster using pysheds
grid = Grid.from_raster(dem_5070_path)
dem = grid.read_raster(dem_5070_path)

In [ ]:
# Condition DEM to fill depressions and calculate flow direction
# Fill pits in the DEM
pit_filled_dem = grid.fill_pits(dem)

# Fill depressions in the DEM
flood_dem = grid.fill_depressions(pit_filled_dem)

# Resolve flats in the DEM
inflated_dem = grid.resolve_flats(flood_dem)


In [ ]:
# Determmine D8 flow direction from DEM
# Specify directional mapping
dirmap = (64, 128, 1, 2, 4, 8, 16, 32)

# Compute flow directions 
fdir = grid.flowdir(inflated_dem, dirmap=dirmap)

In [ ]:
# Replace 0 values with NaN for better visualization
fdir_plot = np.where(fdir == 0, np.nan, fdir)

# Plot flow direction with watershed boundary on top
fig, ax = plt.subplots(figsize=(8,6))

# Plot flow direction with a colormap and colorbar with label
im = ax.imshow(
    fdir_plot,
    extent=grid.extent,
    cmap='viridis',
    zorder=2 # Set zorder to plot below the watershed boundary
)

# Add watershed boundary on top
nisqually_gdf_5070.boundary.plot(ax=ax, edgecolor="white")

# Add colorbar with ticks corresponding to flow direction values
boundaries = [0] + sorted(list(dirmap))
plt.colorbar(im, boundaries=boundaries, values=sorted(dirmap))

# Add title and labels
ax.set_title('Flow direction grid', size=14)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.grid(zorder=-1) # Add gridlines behind the plot

plt.tight_layout()
plt.show()

In [ ]:
# Calcuate flow accumulation
acc = grid.accumulation(fdir, dirmap=dirmap)

In [ ]:
# Save the flow accumulation raster
flow_acc_path = os.path.join(topography_dir, "nisqually_flow_acc_5070.tif")
grid.to_raster(acc, flow_acc_path)

# Show the path to the saved flow accumulation raster
print("Flow accumulation raster saved to:", flow_acc_path)

In [ ]:
# Open saved 5070 flow accumulation raster
nisqually_flow_acc = rxr.open_rasterio(flow_acc_path, masked=True).squeeze()

# Mask zeros for log scale
acc_plot = np.where(nisqually_flow_acc <= 0, np.nan, nisqually_flow_acc)

# Colormap
cmap = plt.get_cmap("cubehelix").copy()
cmap.set_bad("white")

# Bounds for imshow need order: left, right, bottom, top
left, bottom, right, top = nisqually_flow_acc.rio.bounds()

# Plot flow accumulation with log scale
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_alpha(0)
ax.grid(zorder=0)

im = ax.imshow(
    acc_plot,
    extent=(left, right, bottom, top),
    cmap=cmap,
    norm=colors.LogNorm(vmin=1, vmax=np.nanmax(acc_plot)),
    interpolation="bilinear",
    zorder=2
)

# Add watershed boundary on top
nisqually_gdf_5070.boundary.plot(ax=ax, edgecolor="white")

plt.colorbar(im, ax=ax, label="Upstream Cells (log scale)")
ax.set_title("Flow Accumulation (EPSG:5070)", size=14)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

### 3.2 Climate Data Processing

#### 3.2.1 Mean Wet-Season Precipitation

In [ ]:
# Historical files only
hist_files = sorted([
    str(f) for f in saved_files
    if "maca_CCSM4_historical_Nisqually" in Path(f).name
])

# Future files only
fut_files = sorted([
    str(f) for f in saved_files
    if "maca_CCSM4_rcp45_Nisqually" in Path(f).name
])

# Quick check of the files
print(len(hist_files), len(fut_files))
print(hist_files[:2])
print(fut_files[:2])

# Open the historical and future datasets as xarray multi-file datasets
ds_hist = xr.open_mfdataset(hist_files, combine="by_coords")
ds_fut = xr.open_mfdataset(fut_files, combine="by_coords")

In [ ]:
# Check the data variables in each dataset to see if it's "pr" or "precipitation"
pr_hist = ds_hist["pr"] if "pr" in ds_hist.data_vars else ds_hist["precipitation"]
pr_fut  = ds_fut["pr"] if "pr" in ds_fut.data_vars else ds_fut["precipitation"]

In [ ]:
# Subset to years 1976-2005
pr_hist = pr_hist.sel(time=slice("1976-01-01", "2005-12-31"))

# Subset to years 2041-2070
pr_fut = pr_fut.sel(time=slice("2041-01-01", "2070-12-31"))

In [ ]:
# Check out that it worked 
print(pr_hist.time.min().values, pr_hist.time.max().values)
print(len(pr_hist.time))

In [ ]:
# Create function to calculate wet season mean 
def wet_season_mean_raster(da):
    """
    Calculate mean daily precipitation during the wet season (DJF).

    Args:
        da (xarray.DataArray): Daily precipitation with time dimension

    Returns:
        xarray.DataArray: Mean DJF precipitation raster (mm/day)
    """
    da_wet = da.sel(time=da["time.season"] == "DJF")
    return da_wet.mean(dim="time")

In [ ]:
# Calculate the wet season mean raster - historical
pr_hist_wet_mean = wet_season_mean_raster(pr_hist)

# Calculate the wet season mean raster - future
pr_future_wet_mean = wet_season_mean_raster(pr_fut)

In [ ]:
# Create figure and axis for plotting historical
fig, ax = plt.subplots(figsize=(7, 6))

# Plot the historical wet-season mean precipitation raster
pr_hist_wet_mean.plot(
    ax=ax,
    cmap="Blues", # Blue color scale
    cbar_kwargs={"label": "Mean DJF precipitation (mm/day)"} # Add label
)

# Convert watershed boundary to match raster CRS (EPSG:4326)
# and overlay it on the map for spatial reference
nisqually_4326 = nisqually_gdf.to_crs("EPSG:4326")
nisqually_4326.boundary.plot(ax=ax, color="black", linewidth=1)

# Add a visual buffer around the raster extent
buffer = 0.1

# Expand x-axis limits
ax.set_xlim(
    float(pr_hist_wet_mean.lon.min()) - buffer,
    float(pr_hist_wet_mean.lon.max()) + buffer
)

# Expand y-axis limits
ax.set_ylim(
    float(pr_hist_wet_mean.lat.min()) - buffer,
    float(pr_hist_wet_mean.lat.max()) + buffer
)

# Add a descriptive title for the map
ax.set_title("Historical Mean Wet-Season Precipitation (1976–2005)")

# Display the plot
plt.show()

In [ ]:
# Create figure and axis for plotting future
fig, ax = plt.subplots(figsize=(7, 6))

# Plot the future wet-season mean precipitation raster
pr_future_wet_mean.plot(
    ax=ax,
    cmap="Blues",  # Blue color scale
    cbar_kwargs={"label": "Mean DJF precipitation (mm/day)"}  # Add label
)

# Convert watershed boundary to match raster CRS (EPSG:4326)
# and overlay it on the map for spatial reference
nisqually_4326 = nisqually_gdf.to_crs("EPSG:4326")
nisqually_4326.boundary.plot(ax=ax, color="black", linewidth=1)

# Add a visual buffer around the raster extent
buffer = 0.1

# Expand x-axis limits (longitude)
ax.set_xlim(
    float(pr_future_wet_mean.lon.min()) - buffer,
    float(pr_future_wet_mean.lon.max()) + buffer
)

# Expand y-axis limits (latitude)
ax.set_ylim(
    float(pr_future_wet_mean.lat.min()) - buffer,
    float(pr_future_wet_mean.lat.max()) + buffer
)

# Add a descriptive title for the map
ax.set_title("Future Mean Wet-Season Precipitation (2041–2070)")

# Display the plot
plt.show()

In [ ]:
# Calculate the difference between future and historical wet-season precipitation
# Positive values indicate an increase in precipitation
delta_pr = pr_future_wet_mean - pr_hist_wet_mean

# Create figure and axis for plotting the difference map
fig, ax = plt.subplots(figsize=(7, 6))

# Plot the difference raster
delta_pr.plot(
    ax=ax,
    cmap="RdBu", # Diverging colormap  (red = decrease, blue = increase)
    vmin=-1.2, # Set color scale evenly
    vmax=1.2,  # Set color scale evenly
    cbar_kwargs={"label": "Change in precipitation (mm/day)"}
)

# Add a descriptive title explaining the comparison
ax.set_title("Change in Mean Wet-Season Precipitation\n(2041–2070 minus 1976–2005)")

# Overlay watershed boundary for spatial reference
nisqually_gdf.to_crs("EPSG:4326").boundary.plot(
    ax=ax,
    color="black",
    linewidth=1
)

# Display the plot
plt.show()

In [ ]:
# Double check min / max values
print(float(delta_pr.min()))
print(float(delta_pr.max()))

#### 3.2.2 Rx1day (Maximum 1-Day Precipitation)

In [ ]:
# Historical Rx1day calculations
# Subset to highest day in a year
annual_max_hist = pr_hist.resample(time="YE").max()

# Take the mean of all the years Rx1day
rx1day_hist_mean = annual_max_hist.mean(dim="time")

# Future Rx1day calculations
# Subset to highest day in a year
annual_max_fut = pr_fut.resample(time="YE").max()

# Take the mean of all the years Rx1day
rx1day_fut_mean = annual_max_fut.mean(dim="time")

In [ ]:
# Check out the min/max to make sure the values make sense
print("Historical Rx1day min/max:")
print(float(rx1day_hist_mean.min()), float(rx1day_hist_mean.max()))

print("Future Rx1day min/max:")
print(float(rx1day_fut_mean.min()), float(rx1day_fut_mean.max()))

In [ ]:
# Create figure and axis for plotting historical Rx1day
fig, ax = plt.subplots(figsize=(7, 6))

# Plot the historical Rx1day raster
rx1day_hist_mean.plot(
    ax=ax,
    cmap="Purples",  # Good for extreme values
    cbar_kwargs={"label": "Rx1day (mm/day)"}  # Add label
)

# Overlay watershed boundary
nisqually_4326 = nisqually_gdf.to_crs("EPSG:4326")
nisqually_4326.boundary.plot(ax=ax, color="black", linewidth=1)

# Add title
ax.set_title("Historical Rx1day (1976–2005)")

# Display plot
plt.show()

In [ ]:
# Create figure and axis for plotting future Rx1day
fig, ax = plt.subplots(figsize=(7, 6))

# Plot the future Rx1day raster
rx1day_fut_mean.plot(
    ax=ax,
    cmap="Purples",
    cbar_kwargs={"label": "Rx1day (mm/day)"}
)

# Overlay watershed boundary
nisqually_4326.boundary.plot(ax=ax, color="black", linewidth=1)

# Add title
ax.set_title("Future Rx1day (2041–2070)")

# Display plot
plt.show()

In [ ]:
# Calculate the difference between future and historical wet-season precipitation
# Positive values indicate an increase in precipitation
change_pr_max = rx1day_fut_mean - rx1day_hist_mean

# Create figure and axis for plotting the difference map
fig, ax = plt.subplots(figsize=(7, 6))

# Plot the difference raster
change_pr_max.plot(
    ax=ax,
    cmap="RdBu", # Diverging colormap  (red = decrease, blue = increase)
    vmin=-10, # Set color scale evenly
    vmax=10,  # Set color scale evenly
    cbar_kwargs={"label": "Change in precipitation (mm/day)"}
)

# Add a descriptive title explaining the comparison
ax.set_title("Change in Rx1day Mean Precipitation - Intensity Proxy \n(2041–2070 minus 1976–2005)")

# Overlay watershed boundary for spatial reference
nisqually_gdf.to_crs("EPSG:4326").boundary.plot(
    ax=ax,
    color="black",
    linewidth=1
)

# Display the plot
plt.show()

In [ ]:
# Create function to prepare raster for export
# *Created with help of ChatGPT, refined by author*
def prep_raster_for_export(da, x_dim="lon", y_dim="lat", crs="EPSG:4326"):
    """
    Set spatial dimensions and CRS on a DataArray before export.

    Args:
        da (xarray.DataArray): DataArray to prepare
        x_dim (str): Name of longitude dimension. Default is "lon".
        y_dim (str): Name of latitude dimension. Default is "lat".
        crs (str): Coordinate reference system to set. Default is "EPSG:4326".

    Returns:
        xarray.DataArray: Prepared DataArray with spatial metadata
    """
    # Set spatial dimensions and CRS using rioxarray
    da = da.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim)
    da = da.rio.write_crs(crs)
    return da

In [ ]:
# Calculate the paths for all climate rasters to be exported
# Calculate mean wet season precipitation
pr_hist_wet_mean = prep_raster_for_export(pr_hist_wet_mean)
pr_fut_wet_mean = prep_raster_for_export(pr_future_wet_mean)

# Calculate mean Rx1day
rx1day_hist_mean = prep_raster_for_export(rx1day_hist_mean)
rx1day_fut_mean = prep_raster_for_export(rx1day_fut_mean)

In [ ]:
# Check that the spatial dimensions and CRS are set correctly
print(pr_hist_wet_mean.rio.crs)
print(pr_hist_wet_mean.rio.x_dim, pr_hist_wet_mean.rio.y_dim)

In [ ]:
# Create function to write raster
def write_raster(da, out_path):
    """
    Save a climate xarray DataArray as a GeoTIFF raster.

    Args:
        da (xarray.DataArray): Raster to export
        out_path (str): File path for output GeoTIFF

    Returns:
        None
    """

    # Save the calculated raster to a GeoTIFF file
    da.rio.to_raster(out_path)
    print(f"Saved: {out_path}")

In [ ]:
# Save the rasters as GeoTIFFs for plotting
write_raster(pr_hist_wet_mean, os.path.join(climate_dir, "pr_hist_djf_mean_4326.tif"))
write_raster(pr_future_wet_mean, os.path.join(climate_dir, "pr_fut_djf_mean_4326.tif"))
write_raster(rx1day_hist_mean, os.path.join(climate_dir, "rx1day_hist_mean_4326.tif"))
write_raster(rx1day_fut_mean, os.path.join(climate_dir, "rx1day_fut_mean_4326.tif"))

### 3.3 Data Harmonization

#### 3.3.1 Define Reference Raster

A reference raster was established to ensure all model inputs shared a common projection, extent, resolution, and grid alignment prior to fuzzy logic modeling.

In [ ]:
# Open one of the saved rasters to create reference raster for harmonization
reference_raster_dem = rxr.open_rasterio(
    os.path.join(topography_dir, "nisqually_dem_5070.tif")
).squeeze()

In [ ]:
# Load and inspect all rasters, comparing to the reference DEM raster 
reference_raster_dem.name = "reference_dem"

In [ ]:
# Check the shape, CRS, resolution, bounds, and name of the reference raster
print(reference_raster_dem.shape) # Shape (height, width)
print(reference_raster_dem.rio.crs) # CRS check
print(reference_raster_dem.rio.resolution()) # Resolution (pixel size in x and y)
print(reference_raster_dem.rio.bounds()) # Bounds (left, bottom, right, top)
print(reference_raster_dem.name) # Name of the DataArray

#### 3.3.2 Inspect Input Rasters

In [ ]:
# Create function to load and inspect a raster comparison to reference raster
# *Created with help of ChatGPT, refined by author*
def load_and_inspect_raster(path, name=None, default_crs=None, reference=None):
    """
    Load a single-band raster and inspect metadata important for harmonization.

    Args:
        path (str): Full path to raster file.
        name (str, optional): Custom name to assign to the raster.
        default_crs (str, optional): CRS to assign only if missing.
        reference (xarray.DataArray, optional): Reference raster for comparison.

    Returns:
        xarray.DataArray
    """

    # Open raster with rioxarray and squeeze to 2D if it has a band dimension
    da = rxr.open_rasterio(path).squeeze()

    # Assign name if provided
    if name is not None:
        da.name = name

    # Check if CRS is missing and assign default if provided
    if da.rio.crs is None:
        if default_crs is None:
            print(f"WARNING: {name or path} has no CRS assigned.")
        else:
            da = da.rio.write_crs(default_crs)
            print(f"WARNING: {name or path} had no CRS. Assigned {default_crs}.")

    # Inspect raster metadata
    print("\n----- Raster Inspection -----")
    print(f"Name:       {da.name}") # Name of the DataArray
    print(f"Shape:      {da.shape}") # Shape (height, width)
    print(f"Dimensions: {da.dims}") # Check dimensions
    print(f"CRS:        {da.rio.crs}") # CRS check
    print(f"Resolution: {da.rio.resolution()}") # Resolution (pixel size in x and y)
    print(f"Bounds:     {da.rio.bounds()}") # Bounds (left, bottom, right, top)
    print(f"Dtype:      {da.dtype}") # Data type of the raster values
    print(f"NoData:     {da.rio.nodata}") # NoData value

    # Compare to reference raster
    if reference is not None:
        print("\n--- Compared to Reference ---")
        print(f"Reference:       {reference.name}") # Name of the reference raster
        print(f"Same CRS:        {da.rio.crs == reference.rio.crs}") # CRS match
        print(f"Same Shape:      {da.shape == reference.shape}") # Shape match
        print(f"Same Resolution: {da.rio.resolution() == reference.rio.resolution()}") # Resolution match
        print(f"Same Bounds:     {da.rio.bounds() == reference.rio.bounds()}") # Bounds/extent match

    # Return the loaded DataArray
    return da

In [ ]:
# Create a dictionary of raster names and paths for loading and inspection
rasters = {
    "dem": os.path.join(topography_dir, "nisqually_dem_5070.tif"),
    "slope": os.path.join(topography_dir, "nisqually_slope_5070.tif"),
    "flow_acc": os.path.join(topography_dir, "nisqually_flow_acc_5070.tif"),
    "impervious": os.path.join(impervious_dir, "nisqually_impervious_5070.tif"),
    "soil_score": os.path.join(soils_dir, "nisqually_soil_score_5070.tif"),
    "pr_hist_djf_mean": os.path.join(climate_dir, "pr_hist_djf_mean_4326.tif"),
    "pr_fut_djf_mean": os.path.join(climate_dir, "pr_fut_djf_mean_4326.tif"),
    "rx1day_hist_mean": os.path.join(climate_dir, "rx1day_hist_mean_4326.tif"),
    "rx1day_fut_mean": os.path.join(climate_dir, "rx1day_fut_mean_4326.tif"),
}

# Create an empty dictionary to store the loaded rasters
loaded = {}

# Loop through the rasters, load and inspect each one, and store in the loaded dictionary
for name, path in rasters.items():
    if not os.path.exists(path):
        print(f"WARNING: File not found for {name}: {path}")
        continue

    # Use the load_and_inspect_raster to compare to the reference DEM raster
    loaded[name] = load_and_inspect_raster(
        path=path,
        name=name,
        reference=reference_raster_dem
    )

#### 3.3.3 Harmonize Input Rasters

In [ ]:
# Create a function to harmonize climate rasters to match the reference DEM raster
def harmonize_to_reference(path, name, reference_dem, input_crs="EPSG:4326", resampling_method=Resampling.bilinear):
    """Function to load the climate rasters, check/assign CRS, and reproject
    to match the reference DEM raster.
    
    Args:
        path (str): Path to the input climate raster.
        name (str): Name to assign to the loaded DataArray.
        reference_dem (xarray.DataArray): Reference DEM raster to match.
        input_crs (str): CRS to assign if the input raster is missing CRS. Default is "EPSG:4326".
        
    Returns:
        xarray.DataArray: Harmonized climate raster matching the reference DEM."""
    
    da = rxr.open_rasterio(path).squeeze()
    da.name = name

    if da.rio.crs is None:
        da = da.rio.write_crs(input_crs)
        print(f"Assigned missing CRS {input_crs} to {name}")

    da_harmonized = da.rio.reproject_match(
        reference_dem,
    resampling=resampling_method    
    )

    da_harmonized.name = name

    if da_harmonized.ndim == 3:
        da_harmonized = da_harmonized.squeeze()

    return da_harmonized

In [ ]:
# Define which rasters are continuous vs categorical for choosing resampling method
continuous_vars = [
    "slope",
    "flow_acc",
    "impervious",
    "pr_hist_djf_mean",
    "pr_fut_djf_mean",
    "rx1day_hist_mean",
    "rx1day_fut_mean"
]

# Categorigal variable
categorical_vars = [
    "soil_score"
]

# Create a new dictionary to store the harmonized rasters, starting with the DEM which is already the reference
harmonized_rasters = {
    "dem": loaded["dem"]
}

for name in continuous_vars + categorical_vars:
    print("Processing:", name)
    if name in continuous_vars:
        resampling_method = Resampling.bilinear
    else:
        resampling_method = Resampling.nearest

    harmonized_rasters[name] = harmonize_to_reference(
        path=rasters[name],
        name=name,
        reference_dem=reference_raster_dem,
        input_crs="EPSG:4326" if name in [
            "pr_hist_djf_mean",
            "pr_fut_djf_mean",
            "rx1day_hist_mean",
            "rx1day_fut_mean"
        ] else None,
        resampling_method=resampling_method
    )

#### 3.3.4 Verify Harmonized Outputs

In [ ]:
# Loop through the harmonized rasters and inspect metadata, comparing to the reference DEM raster
for name, da in harmonized_rasters.items():

    print("\nRaster Inspection")
    print(f"Name:       {name}")
    print(f"Shape:      {da.shape}")
    print(f"CRS:        {da.rio.crs}")
    print(f"Resolution: {da.rio.resolution()}")
    print(f"Bounds:     {da.rio.bounds()}")

    print("\nCompared to Reference")
    print(f"Same CRS:        {da.rio.crs == reference_raster_dem.rio.crs}")
    print(f"Same Shape:      {da.shape == reference_raster_dem.shape}")
    print(f"Same Resolution: {da.rio.resolution() == reference_raster_dem.rio.resolution()}")
    print(f"Same Bounds:     {da.rio.bounds() == reference_raster_dem.rio.bounds()}")

#### 3.3.5 Save Harmonized Rasters

In [ ]:
# Make inputs folder for final harmonized rasters
inputs_dir = os.path.join(project_dir, "final_inputs_5070")
os.makedirs(inputs_dir, exist_ok=True)

In [ ]:
# Save all harmonized rasters to the final inputs folder
for name, da in harmonized_rasters.items():
    output_path = os.path.join(inputs_dir, f"{name}_5070.tif")
    da.rio.to_raster(output_path)
    print(f"Saved {name} to {output_path}")

In [ ]:
# Check that all model inputs share the same grid information
for name, da in harmonized_rasters.items():
    print(name, da.shape, da.rio.resolution(), da.rio.crs)

In [ ]:
# Check that raster bounds match the reference DEM
for name, da in harmonized_rasters.items():
    print(name)
    print("Raster bounds:   ", da.rio.bounds())
    print("Reference bounds:", reference_raster_dem.rio.bounds())
    print()

## 4. Fuzzy Logic Modeling

### 4.1 Membership Function Design

The final harmonized model inputs include:
- Slope
- Flow accumulation
- Impervious surface (%)
- Hydrologic soil group runoff score
- Mean wet-season precipitation (historical and future)
- Rx1day (historical and future)

Membership functions were developed for each variable to transform the raw values into fuzzy membership values ranging from 0 to 1.

#### 4.1.1 Selecting Membership Function Breakpoints

Breakpoint selection was informed by published literature, hydrologic interpretation, exploratory data visualization, and iterative sigmoid function testing. Histograms were used to examine the distribution of each variable and evaluate candidate breakpoints where established thresholds were unavailable.

| Variable                      | Basis for Breakpoint Selection                              |
| ----------------------------- | ----------------------------------------------------------- |
| Impervious Surface            | Literature-supported thresholds (5%, 40%, 75%)              |
| Hydrologic Soil Group         | Categorical runoff potential                                |
| Slope                         | Literature and hydrologic interpretation (5°, 15°, 30°)     |
| Flow Accumulation             | Exploratory data visualization and sigmoid function testing |
| Mean Wet-Season Precipitation | Exploratory data visualization and sigmoid function testing |
| Rx1day                        | Exploratory data visualization and sigmoid function testing |


In [ ]:
# Create variables list for histogram plotting
variables = [
    "impervious",
    "soil_score",
    "slope",
    "flow_acc",
    "pr_hist_djf_mean",
    "pr_fut_djf_mean",
    "rx1day_hist_mean",
    "rx1day_fut_mean"
]

# Create readable titles for histogram plots
titles = [
    "Percent Impervious Surface",
    "Soil Runoff Score",
    "Slope",
    "Flow Accumulation",
    "Wet Season Precipitation - Historical",
    "Wet Season Precipitation - Future",
    "Rx1day - Historical",
    "Rx1day - Future"
]

# Set axis 3 columns 3 rows for histograms
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

# Loop through the harmonized rasters and plot histograms of the values for each variable, skipping the DEM
for ax, var, title in zip(axes, variables, titles):
    da = harmonized_rasters[var]
    data = da.values

    # Deal with NoData values by masking them out of the data array before plotting
    nodata = da.rio.nodata
    if nodata is not None and not np.isnan(nodata):
        data = data[data != nodata]

    # Mask out any NaN values
    data = data[~np.isnan(data)]

    # Plot histogram of the data values
    ax.hist(data, bins=50, edgecolor="black")
    ax.set_title(title)
    ax.set_xlabel(var)
    ax.set_ylabel("Cell count")

# Hide any unused subplots
for ax in axes[len(variables):]:
    ax.set_visible(False)

# Show the histograms
plt.tight_layout()
plt.show()

In [ ]:
# Print summary statistics for each variable
for var in variables:
    
    # Get the data array and values for the variable
    da = harmonized_rasters[var]
    data = da.values

    # Deal with nodata by masking them
    nodata = da.rio.nodata

    if nodata is not None and not np.isnan(nodata):
        data = data[data != nodata]

    data = data[~np.isnan(data)]

    # Print the percentiles 
    print(f"\n{var}")
    print("Min:", np.min(data))
    print("P25:", np.percentile(data, 25))
    print("P50:", np.percentile(data, 50))
    print("P75:", np.percentile(data, 75))
    print("P90:", np.percentile(data, 90))
    print("P95:", np.percentile(data, 95))
    print("Max:", np.max(data))

In [ ]:
# Log transform flow accumulation for better visualization and reduce skew
flow = harmonized_rasters["flow_acc"].values
flow = flow[~np.isnan(flow)]

# Plot histogram of log-transformed flow accumulation
plt.hist(np.log1p(flow), bins=50, edgecolor="black")
plt.title("Log-transformed Flow Accumulation")
plt.xlabel("log(flow accumulation + 1)")
plt.ylabel("Cell count")
plt.show()

In [ ]:
# Percentiles of log-transformed flow accumulation
flow = harmonized_rasters["flow_acc"].values

# Mask out NoData and NaN values
flow = flow[flow > 0]

# Log transform the flow accumulation values for better visualization and to reduce skew
log_flow = np.log1p(flow)

# Loop for percentiles and print
for p in [25, 50, 75, 90, 95]:
    print(p, np.percentile(log_flow, p))

#### 4.1.2 Testing Membership Functions

Testing Membership function values, b = midpoint, c = steepness of curve

In [ ]:
# Run a test of the sigmoid membership function with some example values
test_values = np.array([0, 15, 40, 75, 100])

# Test the midpoint breakpoint to see if the membership function is working as expected
test_membership = fuzz.sigmf(
    test_values,
    b=30, # Midpoint breakpoint for the sigmoid function
    c=0.12 # Steepness of the curve (higher values = steeper curve)
)

# Calculate the membership values for the test values and print them rounded to 3 decimal places
for value, membership in zip(test_values, test_membership): 
    print(value, round(membership, 3))

In [ ]:
# Test different midpoints to see how the membership function changes
test_values = np.array([0, 10, 15, 25, 30, 40, 75, 100])

# Test different midpoints for the sigmoid membership function and print the results
for b in [25, 30, 40]:
    memberships = fuzz.sigmf(test_values, b=b, c=0.12)
    print(f"\nb = {b}")
    for value, membership in zip(test_values, memberships):
        print(value, round(membership, 3))

In [ ]:
# Plot the sigmoid membership function for different midpoints to visualize how it changes
x = np.arange(0, 101, 1)

# Plot the sigmoid membership function for different midpoints to visualize how it changes
plt.plot(x, fuzz.sigmf(x, b=25, c=0.12), label="b=25")
plt.plot(x, fuzz.sigmf(x, b=30, c=0.12), label="b=30")
plt.plot(x, fuzz.sigmf(x, b=40, c=0.12), label="b=40")

plt.axvline(10, linestyle="--")
plt.axvline(30, linestyle="--")
plt.axvline(75, linestyle="--")

plt.xlabel("Impervious surface (%)")
plt.ylabel("Fuzzy runoff membership")
plt.legend()
plt.show()

In [ ]:
# Test different c values (steepness of the curve)
for c in [0.08, 0.10, 0.12, 0.15]:
    plt.plot(
        x,
        fuzz.sigmf(x, b=30, c=c), #midpoint constant, varying steepness
        label=f"c={c}"
    )

plt.axvline(10, linestyle="--")
plt.axvline(30, linestyle="--")
plt.axvline(75, linestyle="--")

plt.legend()
plt.show()

### 4.2 Fuzzification

#### 4.2.1 Fuzzy Membership Functions


In [ ]:
# Create function for masking no data in rasters
def mask_raster_nodata(raster):
    """
    Mask NoData values in a raster while preserving valid zero values.

    Args:
        raster (xarray.DataArray): Input raster with NoData values.

    Returns:
        xarray.DataArray: Raster with NoData values masked as NaN.
    """

    # Get the NoData value from the raster metadata
    nodata = raster.rio.nodata

    # If nodata is NaN, mask out NaN values    
    if np.isnan(nodata):
        raster_masked = raster.where(~np.isnan(raster))

    # If nodata is -9999, mask out -9999 values
    elif nodata == -9999:
        raster_masked = raster.where(raster != -9999)

    # Leave the raster unchanged to avoid masking valid zero values.    
    else:
        raster_masked = raster

    return raster_masked

In [ ]:
# Create function to convert raster to fuzzy membership with sigmoid function
def fuzzy_sigmoid_increasing(raster, breakpoint, steepness, output_name):
    """
    Convert a raster to fuzzy membership using an increasing sigmoid function.

    Higher input values result in higher fuzzy membership values.

    Args:
        raster (xarray.DataArray): Input raster
        breakpoint (float): Sigmoid midpoint where membership equals 0.5
        steepness (float): Controls how quickly membership increases
        output_name (str): Name for the output fuzzy raster

    Returns:
        xarray.DataArray: Fuzzy raster with membership values from 0 to 1
    """

    # Mask NoData values before fuzzifying
    raster_masked = mask_raster_nodata(raster)

    # Apply increasing sigmoid membership function
    fuzzy_values = fuzz.sigmf(
        raster_masked.values,
        b=breakpoint,
        c=steepness
    )

    # Convert NumPy array back to xarray DataArray
    fuzzy_raster = xr.DataArray(
        fuzzy_values,
        coords=raster_masked.coords,
        dims=raster_masked.dims,
        name=output_name
    )

    # Verify fuzzy membership values are between 0 and 1
    assert float(fuzzy_raster.min()) >= 0, "Membership values are below 0."
    assert float(fuzzy_raster.max()) <= 1, "Membership values are above 1."

    return fuzzy_raster

In [ ]:
# Create function for testing sigmoid membership function with different breakpoints and steepness values
def test_sigmoid_membership(
    breakpoint,
    steepness,
    lower_breakpoint,
    upper_breakpoint,
    x_min,
    x_max,
    title,
):
    """
    Plot an increasing fuzzy sigmoid membership function and display
    membership values at key breakpoints.

    Args:
        breakpoint (float): Midpoint of the sigmoid (membership = 0.5) (b)
        steepness (float): Controls how quickly membership increases (c)
        lower_breakpoint (float): Lower threshold
        upper_breakpoint (float): Upper threshold
        x_min (float): Minimum x-axis value
        x_max (float): Maximum x-axis value
        title (str): Plot title
    """

    # Create x values for plotting
    x = np.linspace(x_min, x_max, 200)

    # Calculate fuzzy membership values
    membership = fuzz.sigmf(
        x,
        b=breakpoint,
        c=steepness
    )

    # Plot sigmoid curve
    plt.figure(figsize=(6, 4))

    plt.plot(
        x,
        membership,
        label=f"b={breakpoint}, c={steepness}"
    )

    # Plot breakpoint lines
    plt.axvline(
        lower_breakpoint,
        linestyle="--",
        color="gray",
        label="Lower breakpoint"
    )

    plt.axvline(
        breakpoint,
        linestyle="--",
        color="black",
        label="Midpoint"
    )

    plt.axvline(
        upper_breakpoint,
        linestyle="--",
        color="red",
        label="Upper breakpoint"
    )

    plt.xlabel("Variable value")
    plt.ylabel("Fuzzy membership")
    plt.title(title)
    plt.ylim(0, 1.05)
    plt.legend()
    plt.show()

    # Print membership values at key points
    test_values = np.array([
        x_min,
        lower_breakpoint,
        breakpoint,
        upper_breakpoint,
        x_max
    ])

    # Check the values of the fuzzy membership function at the test points
    test_membership = fuzz.sigmf(
        test_values,
        b=breakpoint,
        c=steepness
    )
    # Print the test values and their corresponding membership values
    print("Value : Membership")

    # Calculate and print the membership values for the test values, formatted to 3 decimal places
    # *Created with help of ChatGPT, refined by author*
    for value, membership in zip(test_values, test_membership):
        print(f"{value} : {membership:.3f}")

In [ ]:
# Create raster that clips fuzzy raster to watershed
def clip_raster_to_watershed(raster, watershed_gdf, output_name=None):
    """
    Clip a raster to the watershed boundary

    Args:
        raster (xarray.DataArray): Raster to clip
        watershed_gdf (geopandas.GeoDataFrame): Watershed boundary
        output_name (str, optional): Name for the clipped raster

    Returns:
        xarray.DataArray: Raster clipped to the watershed boundary
    """

    clipped_raster = raster.rio.clip(
        watershed_gdf.geometry,
        watershed_gdf.crs,
        drop=False
    )

    if output_name is not None:
        clipped_raster.name = output_name

    return clipped_raster

In [ ]:
# Create function to save the raster
def save_raster(raster, output_dir, filename):
    """
    Save a raster to a GeoTIFF raster file 

    Args:
        raster (xarray.DataArray): Raster to save
        output_dir (str): Folder where output raster will be saved
        filename (str): Output filename
    """

    # Create output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)

    # Build output file path
    output_path = os.path.join(output_dir, filename)

    # Save raster
    raster.rio.to_raster(output_path)

    print(f"Saved raster to: {output_path}")

#### 4.2.2 Impervious Surface

In [ ]:
# Run the fuzzy function for impervious surface with specified parameters
impervious_fuzzy = fuzzy_sigmoid_increasing(
    raster=harmonized_rasters["impervious"],
    breakpoint=30, # Midpoint for the sigmoid function
    steepness=0.12, # Steepness of the curve
    output_name="impervious_fuzzy"
)

# Plot it
impervious_fuzzy.plot()

In [ ]:
# Save impervious raster 
save_raster(
    raster=impervious_fuzzy,
    output_dir=fuzzy_dir,
    filename="impervious_fuzzy.tif"
)

#### 4.2.3 Slope


In [ ]:
# Run the test function for slope
test_sigmoid_membership(
    breakpoint=15,
    steepness=0.15,
    lower_breakpoint=5,
    upper_breakpoint=30,
    x_min=0,
    x_max=60,
    title="Slope fuzzy membership function"
)

In [ ]:
slope_fuzzy = fuzzy_sigmoid_increasing(
    raster=harmonized_rasters["slope"],
    breakpoint=15,
    steepness=0.1,
    output_name="slope_fuzzy"
)

slope_fuzzy.plot()

In [ ]:
# Clip the slope fuzzy raster to the watershed boundary
slope_fuzzy_clipped = clip_raster_to_watershed(
    raster=slope_fuzzy,
    watershed_gdf=nisqually_gdf_5070
)

# Plot it 
slope_fuzzy_clipped.plot()

In [ ]:
# Save the slope fuzzy raster to the fuzzy directory
save_raster(
    raster=slope_fuzzy_clipped,
    output_dir=fuzzy_dir,
    filename="slope_fuzzy_clipped.tif"
)

#### 4.2.4 Flow Accumulation


In [ ]:
flow_acc = harmonized_rasters["flow_acc"]
# Natural log transform, same as np.log1p(flow) used for histogram
flow_acc_log = np.log1p(flow_acc)
flow_acc_log.name = "flow_acc_log"

In [ ]:
# Run the test function for flow accumulation
test_sigmoid_membership(
    breakpoint=1.7,
    steepness=2.7, # Adjust based on the steepness of the curve to high upper/lower breakpoints
    lower_breakpoint=1.2,
    upper_breakpoint=2.2,
    x_min=0,
    x_max=3,
    title="Flow Accumulation fuzzy membership function"
)

In [ ]:
# Run the fuzzy function for flow accumulation with sigmoid parameters
flow_fuzzy = fuzzy_sigmoid_increasing(
    raster=flow_acc_log,
    breakpoint=1.7,
    steepness=2.7,
    output_name="flow_fuzzy"
)

# Plot the flow accumulation fuzzy raster
flow_fuzzy.plot()

In [ ]:
# Clip the flow accumulation fuzzy raster to the watershed boundary
flow_fuzzy_clipped = clip_raster_to_watershed(
    raster=flow_fuzzy,
    watershed_gdf=nisqually_gdf_5070
)

# Plot it 
flow_fuzzy_clipped.plot()

In [ ]:
# Save the flow accumulation fuzzy raster to the fuzzy directory
save_raster(
    raster=flow_fuzzy_clipped,
    output_dir=fuzzy_dir,
    filename="flow_acc_fuzzy_clipped.tif"
)

#### 4.2.5 Mean Wet-Season Precipitation

In [ ]:
# Check out the data
climate = mask_raster_nodata(harmonized_rasters["pr_hist_djf_mean"])

data = climate.values

# Mask out NoData values
data = data[~np.isnan(data)] 

# Calculate the quartiles of the data
q1, q2, q3, = np.percentile(data, [25, 50, 75])

# Print the quartiles
print("Q1:", q1)
print("Q2:", q2)
print("Q3:", q3)


In [ ]:
# Run the test function for climate djf mean precip
test_sigmoid_membership(
    breakpoint=6.3,
    steepness=1.2, # Adjust based on the steepness of the curve to high upper/lower breakpoints
    lower_breakpoint=4.8,
    upper_breakpoint=8.3,
    x_min=0,
    x_max=12,
    title="Historical DJF Precipitation Fuzzy Membership Function"
)

In [ ]:
# Fuzzify the historical mean precip raster
pr_hist_fuzzy = fuzzy_sigmoid_increasing(
    raster=harmonized_rasters["pr_hist_djf_mean"],
    breakpoint=6.3,
    steepness=1.2,
    output_name="pr_hist_djf_fuzzy"
)

# Fuzzify the future mean precip raster
pr_fut_fuzzy = fuzzy_sigmoid_increasing(
    raster=harmonized_rasters["pr_fut_djf_mean"],
    breakpoint=6.3,
    steepness=1.2,
    output_name="pr_fut_djf_fuzzy"
)

In [ ]:
# Plot to check
pr_hist_fuzzy.plot()
   

In [ ]:
# Clip the climate mean djf precip fuzzy raster to the watershed boundary
historical_pr_mean_fuzzy_clipped = clip_raster_to_watershed(
    raster=pr_hist_fuzzy,
    watershed_gdf=nisqually_gdf_5070
)

# Plot it
historical_pr_mean_fuzzy_clipped.plot()

In [ ]:
# Save the future precipitation fuzzy rasters
save_raster(
    raster=pr_hist_fuzzy,
    output_dir=fuzzy_dir,
    filename="pr_hist_djf_fuzzy_clipped.tif"
)

In [ ]:
# Clip the future pr mean djf fuzzy raster to the watershed boundary
future_pr_mean_fuzzy_clipped = clip_raster_to_watershed(
    raster=pr_fut_fuzzy,
    watershed_gdf=nisqually_gdf_5070
)

# Save the future precipitation fuzzy rasters
save_raster(
    raster=future_pr_mean_fuzzy_clipped,
    output_dir=fuzzy_dir,
    filename="pr_fut_djf_fuzzy_clipped.tif"
)


#### 4.2.6 Rx1day

In [ ]:
# Check out the data for rx1day historical mean
climate = mask_raster_nodata(harmonized_rasters["rx1day_hist_mean"])
data = climate.values

# Mask out NoData values
data = data[~np.isnan(data)] 

# Calculate the quartiles of the data
q1, q2, q3 = np.percentile(data, [25, 50, 75])

# Print the quartiles
print("Q1:", q1)
print("Q2:", q2)
print("Q3:", q3)

In [ ]:
# Run the test function for rx1day historical mean precipitation
test_sigmoid_membership(
    breakpoint=59.2,
    steepness=0.15, # Adjust based on the steepness of the curve to high upper/lower breakpoints
    lower_breakpoint=47.47,
    upper_breakpoint=74.74,
    x_min=20,
    x_max=120,
    title="Historical RX1DAY Fuzzy Membership Function"
)

In [ ]:
# Fuzzify the historical mean precip raster
rx1day_hist_fuzzy = fuzzy_sigmoid_increasing(
    raster=harmonized_rasters["rx1day_hist_mean"],
    breakpoint=59.2,
    steepness=0.15,
    output_name="rx1day_hist_fuzzy"
)

# Fuzzify the future mean precip raster
rx1day_fut_fuzzy = fuzzy_sigmoid_increasing(
    raster=harmonized_rasters["rx1day_fut_mean"],
    breakpoint=59.2,
    steepness=0.15,
    output_name="rx1day_fut_fuzzy"
)

In [ ]:
# Plot it to check
rx1day_hist_fuzzy.plot()

In [ ]:
# Clip the historical rx1day mean fuzzy raster to the watershed boundary
historical_rx1day_mean_fuzzy_clipped = clip_raster_to_watershed(
    raster=rx1day_hist_fuzzy,
    watershed_gdf=nisqually_gdf_5070
)

# Save the historical rx1day mean fuzzy rasters
save_raster(
    raster=historical_rx1day_mean_fuzzy_clipped,
    output_dir=fuzzy_dir,
    filename="rx1day_hist_fuzzy_clipped.tif"
)


In [ ]:
historical_rx1day_mean_fuzzy_clipped.plot()

In [ ]:
# Clip the future rx1day mean fuzzy raster to the watershed boundary
future_rx1day_mean_fuzzy_clipped = clip_raster_to_watershed(
    raster=rx1day_fut_fuzzy,
    watershed_gdf=nisqually_gdf_5070
)

# Save the future rx1day mean fuzzy rasters
save_raster(
    raster=future_rx1day_mean_fuzzy_clipped,
    output_dir=fuzzy_dir,
    filename="rx1day_fut_fuzzy_clipped.tif"
)


In [ ]:
# Check it out
future_rx1day_mean_fuzzy_clipped.plot()

### 4.2.7 Hydrologic Soil Groups (Already Fuzzy)

In [ ]:
# Open the fuzzy soil raster
soil_fuzzy = rxr.open_rasterio(
    soil_raster_path,
    masked=True
).squeeze()

# Give it a consistent name
soil_fuzzy.name = "soil_fuzzy"

In [ ]:
# Save the fuzzy soil raster to the fuzzy directory
save_raster(
    raster=soil_fuzzy, 
    output_dir=fuzzy_dir,
    filename="soil_fuzzy_clipped.tif"
)

In [ ]:
# Double check values between 0 and 1
print(float(soil_fuzzy.min()))
print(float(soil_fuzzy.max()))

In [ ]:
soil_fuzzy.plot()

### 4.3 Fuzzy Overlay


Function for Gamma Operator - Fuzzy Logic


In [ ]:
### Function to apply fuzzy gamma overlay to a list of fuzzy rasters
# * created with the help of Claude, refined by author *
def fuzzy_gamma_overlay(layers, gamma=0.9, output_name=None):
    """
    Combine fuzzy rasters using the gamma overlay operator.

    The gamma operator balances between: 
    - Fuzzy algebraic sum (optimistic, driven by strongest layer)
    - Fuzzy algebraic product (pessimistic, driven by weakest layer)

    Args: 
        layers (list of xarray.DataArray): List of fuzzy rasters to combine (0 to 1)
        gamma (float): Gamma parameter (0 < gamma < 1)
            - gamma close to 1: optimistic (algebraic sum dominates)
            - gamma close to 0: pessimistic (algebraic product dominates)
            - 0 < gamma < 1: balance between the two
        output_name (str): Optional name for the output fuzzy raster
    
    Returns:
        xarray.DataArray: Fuzzy susceptibility raster with values 0 to 1
    """

    # Step 1 - Validate inputs 
    # Validate gamma is in valid range
    if not 0 < gamma < 1:
        raise ValueError(f"Gamma must be between 0 and 1, got {gamma}")
    
    # Validate at least two layers are provided
    if len(layers) < 2:
        raise ValueError("At least two fuzzy layers are required for gamma overlay.")
    
    # Validate all layers have the same shape 
    shapes = [layer.shape for layer in layers]
    if len(set(shapes)) > 1:
        raise ValueError("All layers must have the same shape for gamma overlay.")
    
    # Step 2 - Build shared valid pixel mask and masked layers list
    # A pixel is only valid if every single layer has a real value there
    valid_mask = np.ones(layers[0].shape, dtype=bool)
    masked_layers = []

    for layer in layers:
        # Get valid pixels for this layer
        layer_valid = ~np.isnan(layer.values)
                                
        # Update combined mask - pixel must be valid in ALL layers
        valid_mask = valid_mask & layer_valid

        # Add layer to masked layers list
        masked_layers.append(layer)

    # Step 3 - Calculate fuzzy algebraic product and sum
    # Start product at 1 (identity for multiplication)
    # Start sum at 0 (identity for sum formula)
    fuzzy_product = np.ones(layers[0].shape)
    fuzzy_sum = np.zeros(layers[0].shape)

    for layer in masked_layers:
        values = layer.values.copy()

        # Set NaN pixels to neutral values for math
        # Product neutral = 1 (won't affect multiplication)
        # Sum neutral = 0 (won't affect sum)
        values_product = np.where(np.isnan(values), 1.0, values)
        values_sum = np.where(np.isnan(values), 0.0, values)

        # Update fuzzy algebraic product
        fuzzy_product = fuzzy_product * values_product

        # Update fuzzy algebraic sum
        fuzzy_sum = 1 - ((1 - fuzzy_sum) * (1 - values_sum))

    # Step 4 - Apply gamma operator
    gamma_overlay = (fuzzy_sum ** gamma) * (fuzzy_product ** (1 - gamma))

    # Step 5 - Apply valid mask and clip to 0-1
    # Set invalid pixels back to NaN using the shared valid mask
    gamma_overlay = np.where(valid_mask, gamma_overlay, np.nan)
    
    # Clip to 0-1 to handle any floating point drift
    gamma_overlay = np.clip(gamma_overlay, 0, 1)
    
    # Step 6 - Convert back to xarray preserving spatial metadata
    result = xr.DataArray(
        gamma_overlay,
        coords=layers[0].coords,
        dims=layers[0].dims,
        name=output_name
    )
    
    # Write CRS from first layer
    result = result.rio.write_crs(layers[0].rio.crs)
    
    return result

### 5. Results & Maps

#### <u>Model Overview</u>
#### Runoff Generation + Runoff Concentration = Surface Runoff Susceptibility

Rather than combining all variables into a single fuzzy overlay and to better represent hydrological conditions, the model first groups variables according to the hydrologic processes they represent. Runoff generation describes the likelihood that precipitation will become surface runoff, while runoff concentration represents the tendency for runoff to accumulate and converge across the landscape. The variable representing these processes were modeled seperately before being combined to produce the final surface runoff susceptibility map.

Gamma values ranging from 0.5 to 0.9 were evaluated during model development. The gamma parameter controls the balance between the fuzzy algebraic product (more conservative) and fuzzy algebraic sum (more optimistic). A gamma value of 0.85 was selected because it produced results that best represented the combined influence of the input variables while avoiding overly restrictive or overly permissive predictions.

#### 5.1 Runoff Concentration

In [ ]:
### Run the fuzzy gamma overlay for runoff concentration
# Runoff concentration = slope + flow accumulation

runoff_concentration_layers = fuzzy_gamma_overlay(
    layers = [flow_fuzzy_clipped, slope_fuzzy_clipped], 
    gamma=0.85, 
    output_name="runoff_concentration")

# Save the runoff generation raster to the fuzzy directory
save_raster(runoff_concentration_layers, fuzzy_dir, "runoff_concentration_g085.tif")

In [ ]:
# Sanity check for the runoff concentration raster
print("Shape:", runoff_concentration_layers.shape)
print("Min:", float(runoff_concentration_layers.min()))
print("Max:", float(runoff_concentration_layers.max()))
print("NaNs:", int(np.isnan(runoff_concentration_layers.values).sum()))

# Plot it
runoff_concentration_layers.plot(cmap="viridis")
nisqually_gdf_5070.boundary.plot(ax=plt.gca(), edgecolor="black", linewidth=1)
plt.title("Runoff Concentration - Gamma 0.85")
plt.show()

#### 5.2 Runoff Generation 

##### Runoff Generation - Historical Conditions

In [ ]:
### Run the fuzzy gamma overlay for runoff generation - historical precipitation
# Runoff generation = impervious + soil + historical precipitation (rx1day + djf mean)
historical_runoff_generation_layers = fuzzy_gamma_overlay(
    layers = [impervious_fuzzy, 
              soil_fuzzy,
              slope_fuzzy_clipped, 
              historical_pr_mean_fuzzy_clipped,
              historical_rx1day_mean_fuzzy_clipped,], 
    gamma=0.85, 
    output_name="runoff_generation_historical")

# Save the runoff generation raster to the fuzzy directory
save_raster(historical_runoff_generation_layers, fuzzy_dir, "runoff_generation_historical_g085.tif")

In [ ]:
# Sanity check for the runoff generation raster
print("Shape:", historical_runoff_generation_layers.shape)
print("Min:", float(historical_runoff_generation_layers.min()))
print("Max:", float(historical_runoff_generation_layers.max()))
print("NaNs:", int(np.isnan(historical_runoff_generation_layers.values).sum()))

# Plot it
historical_runoff_generation_layers.plot(cmap="viridis")
nisqually_gdf_5070.boundary.plot(ax=plt.gca(), edgecolor="black", linewidth=1)
plt.title("Historical Runoff Generation - Gamma 0.85")
plt.show()

#### Runoff Generation - Future Conditions

In [ ]:
### Run the fuzzy gamma overlay for runoff generation - future precipitation
# Runoff generation = impervious + soil + future precipitation (rx1day + djf mean)
future_runoff_generation_layers = fuzzy_gamma_overlay(
    layers = [impervious_fuzzy, 
              soil_fuzzy, 
              slope_fuzzy_clipped,
              future_pr_mean_fuzzy_clipped,
              future_rx1day_mean_fuzzy_clipped], 
    gamma=0.85, 
    output_name="runoff_generation_future")

save_raster(future_runoff_generation_layers, fuzzy_dir, "runoff_generation_future_g085.tif")

In [ ]:
# Sanity check for the future runoff generation raster
print("Shape:", future_runoff_generation_layers.shape)
print("Min:", float(future_runoff_generation_layers.min()))
print("Max:", float(future_runoff_generation_layers.max()))
print("NaNs:", int(np.isnan(future_runoff_generation_layers.values).sum()))

# Plot it
future_runoff_generation_layers.plot(cmap="viridis")
nisqually_gdf_5070.boundary.plot(ax=plt.gca(), edgecolor="black", linewidth=1)
plt.title("Future Runoff Generation - Gamma 0.85")
plt.show()

In [ ]:
# Quick difference check between historical and future
diff = future_runoff_generation_layers - historical_runoff_generation_layers
print("\nGeneration difference (future - historical):")
print("Min change:", float(diff.min()))
print("Max change:", float(diff.max()))
print("Mean change:", float(diff.mean()))

#### 5.3 Historical Surface Runoff Susceptibility

In [ ]:
### Run the fuzzy gamma overlay for historical runoff susceptibility map
historical_runoff_susceptibility_map = fuzzy_gamma_overlay(
    layers = [runoff_concentration_layers,
              historical_runoff_generation_layers], 
    gamma=0.85, 
    output_name="runoff_susceptibility_historical")

# Save the runoff susceptibility raster to the fuzzy directory
save_raster(historical_runoff_susceptibility_map, final_dir, "runoff_susceptibility_historical_g085.tif")

In [ ]:
# Sanity check for the historical runoff susceptibility map raster
print("Shape:", historical_runoff_susceptibility_map.shape)
print("Shape:", historical_runoff_susceptibility_map.shape)
print("Min:", float(historical_runoff_susceptibility_map.min()))
print("Max:", float(historical_runoff_susceptibility_map.max()))
print("NaNs:", int(np.isnan(historical_runoff_susceptibility_map.values).sum()))

# Plot it
historical_runoff_susceptibility_map.plot(cmap="viridis")
nisqually_gdf_5070.boundary.plot(ax=plt.gca(), edgecolor="black", linewidth=1)
plt.title(
    "Historical Runoff Susceptibility\n"
    "(1976-2005, γ = 0.85)"
)
plt.show()

#### 5.4 Future Surface Runoff Susceptibility

In [ ]:
### Run the fuzzy gamma overlay for future runoff susceptibility map
future_runoff_susceptibility_map = fuzzy_gamma_overlay(
    layers = [runoff_concentration_layers,
              future_runoff_generation_layers], 
    gamma=0.85, 
    output_name="runoff_susceptibility_future")

# Save the runoff susceptibility raster to the fuzzy directory
save_raster(future_runoff_susceptibility_map, final_dir, "runoff_susceptibility_future_g085.tif")

In [ ]:
# Sanity check for the future runoff susceptibility map raster
print("Shape:", future_runoff_susceptibility_map.shape)
print("Shape:", future_runoff_susceptibility_map.shape)
print("Min:", float(future_runoff_susceptibility_map.min()))
print("Max:", float(future_runoff_susceptibility_map.max()))
print("NaNs:", int(np.isnan(future_runoff_susceptibility_map.values).sum()))

# Plot it
future_runoff_susceptibility_map.plot(cmap="viridis")
nisqually_gdf_5070.boundary.plot(ax=plt.gca(), edgecolor="black", linewidth=1)
plt.title(
    "Future Runoff Susceptibility\n"
    "(2041-2070, RCP 4.5, γ = 0.85)"
)
plt.show()

In [ ]:
# Numerical comparison - historical vs future susceptibility
print("Historical Susceptibility:")
print("  Min: ", float(historical_runoff_susceptibility_map.min()))
print("  Max: ", float(historical_runoff_susceptibility_map.max()))
print("  Mean:", float(historical_runoff_susceptibility_map.mean()))

print("\nFuture Susceptibility:")
print("  Min: ", float(future_runoff_susceptibility_map.min()))
print("  Max: ", float(future_runoff_susceptibility_map.max()))
print("  Mean:", float(future_runoff_susceptibility_map.mean()))

print("\nDifference (future - historical):")
diff = future_runoff_susceptibility_map - historical_runoff_susceptibility_map
print("  Min change: ", float(diff.min()))
print("  Max change: ", float(diff.max()))
print("  Mean change:", float(diff.mean()))
print("  Pixels increasing:", int((diff > 0).sum()))
print("  Pixels decreasing:", int((diff < 0).sum()))
print("  Pixels unchanged: ", int((diff == 0).sum()))

#### 5.5 Change Map (Future - Historical)

In [ ]:
# Calculate change map
susceptibility_change = future_runoff_susceptibility_map - historical_runoff_susceptibility_map
susceptibility_change.name = "susceptibility_change"

# Plot change map
susceptibility_change.plot(
    cmap="RdBu_r",
    vmin=-0.05,
    vmax=0.05,
    cbar_kwargs={"label": "Change in susceptibility (future - historical)"}
)
nisqually_gdf_5070.boundary.plot(ax=plt.gca(), edgecolor="black", linewidth=1)
plt.title("Change in Runoff Susceptibility\n(2041-2070 minus 1976-2005) (γ = 0.85)")
plt.show()

# Save to final outputs
save_raster(susceptibility_change, final_dir, "susceptibility_change_g085.tif")

#### 5.6 Interactive Visualization

In [ ]:
# Enable bokeh backend for interactive plotting
gv.extension("bokeh")

# Create interactive map for historical runoff susceptibility
historical_interactive = historical_runoff_susceptibility_map.hvplot.image(
    x="x",
    y="y",
    geo=True,
    crs=ccrs.epsg(5070),
    projection=ccrs.GOOGLE_MERCATOR,
    project=True,
    tiles="CartoLight",
    cmap="viridis",
    clim=(0, 1),
    alpha=0.75,
    colorbar=True,
    clabel="Relative surface runoff susceptibility",
    title="Historical Surface Runoff Susceptibility in the Nisqually Watershed",
    width=750,
    height=600,
    tools=["hover"],
    xaxis=None,
    yaxis=None,
)

# Add watershed boundary overlay
watershed_boundary = nisqually_gdf_5070.hvplot(
    geo=True,
    crs=ccrs.epsg(5070),
    line_color="black",
    line_width=1.5,
    fill_alpha=0
)

# Display the interactive map
historical_interactive * watershed_boundary

In [ ]:
# Enable the Bokeh backend for interactive web maps
gv.extension("bokeh")

# Create an interactive map of future runoff susceptibility
future_interactive = future_runoff_susceptibility_map.hvplot.image(
    x="x",
    y="y",
    geo=True,
    crs=ccrs.epsg(5070),
    projection=ccrs.GOOGLE_MERCATOR,
    project=True,
    tiles="CartoLight",
    cmap="viridis",
    clim=(0, 1),
    alpha=0.75,
    colorbar=True,
    clabel="Relative surface runoff susceptibility",
    title="Future Surface Runoff Susceptibility in the Nisqually Watershed",
    width=800,
    height=650,
    tools=["hover"],
)
# Add watershed boundary overlay
watershed_boundary = nisqually_gdf_5070.hvplot(
    geo=True,
    crs=ccrs.epsg(5070),
    line_color="black",
    line_width=1.5,
    fill_alpha=0
)


# Display the interactive map
future_interactive * watershed_boundary

#### 5.7 Map Stats

In [ ]:
# Calculate change map
diff = future_runoff_susceptibility_map - historical_runoff_susceptibility_map

# Mask NaN values — only count valid watershed pixels
valid_pixels = ~np.isnan(diff.values)

# Count only valid pixels
total_valid = int(valid_pixels.sum())
pixels_increasing = int((diff.values[valid_pixels] > 0).sum())
pixels_decreasing = int((diff.values[valid_pixels] < 0).sum())
pixels_unchanged = int((diff.values[valid_pixels] == 0).sum())

# Calculate percentages stats
pct_increasing_of_changing = pixels_increasing / (pixels_increasing + pixels_decreasing) * 100
pct_changed_of_valid = (pixels_increasing + pixels_decreasing) / total_valid * 100

print(f"Total valid watershed pixels: {total_valid:,}")
print(f"Pixels increasing: {pixels_increasing:,}")
print(f"Pixels decreasing: {pixels_decreasing:,}")
print(f"Pixels unchanged: {pixels_unchanged:,}")
print(f"\n{pct_increasing_of_changing:.1f}% of changing pixels increased")
print(f"{pct_changed_of_valid:.1f}% of valid watershed pixels showed change")

In [ ]:
# Flatten both rasters into 1D arrays and drop any NaN pixels (outside watershed boundary)
hist_vals = historical_runoff_susceptibility_map.values.flatten()
change_vals = susceptibility_change.values.flatten()

mask = ~np.isnan(hist_vals) & ~np.isnan(change_vals)
hist_vals = hist_vals[mask]
change_vals = change_vals[mask]

# Scatter plot: historical susceptibility (x) vs. change (y)
plt.figure(figsize=(8, 6))
plt.scatter(hist_vals, change_vals, alpha=0.1, s=2)
plt.xlabel("Historical Susceptibility")
plt.ylabel("Change (Future − Historical)")
plt.title("Does change concentrate?")
plt.axhline(0, color="gray", linestyle="--", linewidth=0.5)
plt.show()

# Quantify the relationship with a correlation coefficient
correlation = np.corrcoef(hist_vals, change_vals)[0, 1]
print("Correlation between historical susceptibility and change:", correlation)

#### <u>Data Citations</u>
- Abatzoglou, J. T., & Brown, T. J. (2012). A comparison of statistical downscaling methods suited for wildfire applications. International Journal of Climatology, 32(5), 772–780. https://doi.org/10.1002/joc.2313

- NASA Jet Propulsion Laboratory (JPL). (2013). NASA Shuttle Radar Topography Mission Global 1 arc second [Data set]. NASA Land Processes Distributed Active Archive Center. https://doi.org/10.5067/MEASURES/SRTM/SRTMGL1.003

- Soil Survey Staff. Gridded Soil Survey Geographic (gSSURGO) Database for Washington. United States Department of Agriculture, Natural Resources Conservation Service. Available online at https://gdg.sc.egov.usda.gov/. Accessed April 7, 2026.

- U.S. Geological Survey (USGS), 2024, Annual NLCD Collection 1 Science Products: U.S. Geological Survey data release, https://doi.org/10.5066/P94UXNTS

- U.S. Geological Survey. Watershed Boundary Dataset (WBD), 8-digit Hydrologic Unit Code 17110015 — Nisqually. National Geospatial Technical Operations Center, 2025.